In [1]:
# ============================================================
# one model per grade
# Direct multi-output: Dense(60) → no recursive loop
# ============================================================

import os
os.makedirs('saved_models/lstm_v2', exist_ok=True)
os.makedirs('plots/lstm_v2', exist_ok=True)
os.makedirs('results/lstm_v2', exist_ok=True)

print("="*60)
print("   SmartTea AI — FIXED Multi-Output LSTM Training")
print("="*60)
print()
print("   WHY THE FIRST ATTEMPT FAILED:")
print("   → Trained all 5 grades together with 1 scaler")
print("   → Model got confused by different demand levels")
print("   → Result: 30% MAPE (terrible)")
print()
print("   WHAT WE FIX NOW:")
print("   → Train one model per grade (like old approach)")
print("   → Each grade gets its own scaler")
print("   → But output is Dense(60) not Dense(1)")
print("   → So: no recursive loop, no error compounding")
print("   → Expected MAPE: similar to old 7-9%")
print()
print("✅ Setup complete!")

   SmartTea AI — FIXED Multi-Output LSTM Training

   WHY THE FIRST ATTEMPT FAILED:
   → Trained all 5 grades together with 1 scaler
   → Model got confused by different demand levels
   → Result: 30% MAPE (terrible)

   WHAT WE FIX NOW:
   → Train one model per grade (like old approach)
   → Each grade gets its own scaler
   → But output is Dense(60) not Dense(1)
   → So: no recursive loop, no error compounding
   → Expected MAPE: similar to old 7-9%

✅ Setup complete!


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (mean_absolute_error,
                             mean_squared_error)
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (LSTM, Dense,
                                     Dropout,
                                     BatchNormalization)
from tensorflow.keras.callbacks import (EarlyStopping,
                                        ReduceLROnPlateau)
import joblib
import json

plt.style.use('seaborn-v0_8-darkgrid')

print("="*50)
print(f"   TensorFlow : {tf.__version__}")
print(f"   NumPy      : {np.__version__}")
print(f"   pandas     : {pd.__version__}")
print("="*50)
print("✅ Libraries ready!")

   TensorFlow : 2.21.0
   NumPy      : 2.4.6
   pandas     : 3.0.3
✅ Libraries ready!


In [3]:
# ============================================================
# All settings — one place
# ============================================================

LOOK_BACK   = 60    # input: past 60 days
MAX_HORIZON = 60    # output: next 60 days at once

GRADES = ['BOP', 'BOPF', 'DUST', 'FNGS', 'OP']

BATCH_SIZE  = 16    # same as old working model
MAX_EPOCHS  = 120
PATIENCE    = 15

print("="*55)
print("   CONFIGURATION")
print("="*55)
print(f"   Look-back window  : {LOOK_BACK} days")
print(f"   Forecast horizon  : {MAX_HORIZON} days")
print(f"   Grades            : {GRADES}")
print(f"   Batch size        : {BATCH_SIZE}")
print(f"   Max epochs        : {MAX_EPOCHS}")
print(f"   Early stop after  : {PATIENCE} bad epochs")
print()
print("   KEY DESIGN DECISION:")
print("   One model + one scaler PER grade")
print("   Same as old approach but Dense(60) output")
print("="*55)
print()
print("✅ Config ready!")

   CONFIGURATION
   Look-back window  : 60 days
   Forecast horizon  : 60 days
   Grades            : ['BOP', 'BOPF', 'DUST', 'FNGS', 'OP']
   Batch size        : 16
   Max epochs        : 120
   Early stop after  : 15 bad epochs

   KEY DESIGN DECISION:
   One model + one scaler PER grade
   Same as old approach but Dense(60) output

✅ Config ready!


In [4]:
df = pd.read_csv('data/tea_demand_timeseries.csv')
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['TeaGrade','Date']).reset_index(drop=True)

print("="*55)
print("   DATASET LOADED")
print("="*55)
print(f"   Total rows   : {len(df):,}")
print(f"   Date range   : {df['Date'].min().date()} "
      f"→ {df['Date'].max().date()}")
print()

for g in GRADES:
    gdf = df[df['TeaGrade'] == g]
    print(f"   {g:<6}: {len(gdf)} rows | "
          f"avg={gdf['DemandKg'].mean():.1f} kg | "
          f"min={gdf['DemandKg'].min():.0f} | "
          f"max={gdf['DemandKg'].max():.0f}")

print()
print("   Notice: each grade has very different demand levels")
print("   This is WHY one model per grade is required")
print()
print("✅ Data loaded!")

   DATASET LOADED
   Total rows   : 5,475
   Date range   : 2021-01-01 → 2023-12-31

   BOP   : 1095 rows | avg=319.0 kg | min=109 | max=496
   BOPF  : 1095 rows | avg=278.6 kg | min=112 | max=424
   DUST  : 1095 rows | avg=238.4 kg | min=87 | max=361
   FNGS  : 1095 rows | avg=178.7 kg | min=61 | max=281
   OP    : 1095 rows | avg=119.5 kg | min=36 | max=182

   Notice: each grade has very different demand levels
   This is WHY one model per grade is required

✅ Data loaded!


In [5]:
# ============================================================
# Helper functions used in training loop
# ============================================================

def build_sequences(series, look_back, horizon):
    """
    Build (X, y) pairs for direct multi-output.

    X[i] = past 'look_back' days   (input)
    y[i] = next 'horizon'  days    (ALL future at once)

    Example with look_back=60, horizon=60:
    X[0] = days[0:60]    →   y[0] = days[60:120]
    X[1] = days[1:61]    →   y[1] = days[61:121]
    """
    X, y = [], []
    total = len(series) - look_back - horizon + 1
    for i in range(total):
        X.append(series[i : i + look_back])
        y.append(series[i + look_back : i + look_back + horizon])
    return np.array(X), np.array(y)


def build_model(look_back, horizon):
    """
    Build LSTM model for one grade.
    Input shape:  (look_back, 1)
    Output shape: (horizon,)   ← all days at once
    """
    tf.keras.backend.clear_session()

    model = Sequential([

        # Layer 1: LSTM reads 60 days of history
        LSTM(64, return_sequences=True,
             input_shape=(look_back, 1),
             name='LSTM_1'),
        BatchNormalization(name='BN_1'),
        Dropout(0.2, name='Drop_1'),

        # Layer 2: LSTM refines patterns
        LSTM(32, return_sequences=False,
             name='LSTM_2'),
        Dropout(0.2, name='Drop_2'),

        # Layer 3: Dense bridge
        Dense(32, activation='relu', name='Dense_1'),
        Dropout(0.1, name='Drop_3'),

        # Output: ALL 60 days at once
        # THIS is the key upgrade from Dense(1)
        Dense(horizon, name=f'Output_{horizon}days')
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.001),
        loss='mean_squared_error',
        metrics=['mae']
    )
    return model


def mape_score(actual, predicted):
    """Safe MAPE calculation."""
    return np.mean(
        np.abs((actual - predicted) /
               (np.abs(actual) + 1e-8))
    ) * 100


print("✅ Helper functions defined!")
print()
print("   build_sequences() → creates X,y pairs")
print("   build_model()     → creates LSTM per grade")
print("   mape_score()      → evaluation metric")

✅ Helper functions defined!

   build_sequences() → creates X,y pairs
   build_model()     → creates LSTM per grade
   mape_score()      → evaluation metric


In [6]:
# ============================================================
# Main training loop
# Trains 5 models (one per grade)
# Each has its own scaler
# ============================================================

all_grade_results = {}    # store results for summary
all_grade_metadata = {}   # store metadata for API

print("="*65)
print("   TRAINING — ONE MODEL PER GRADE")
print("="*65)
print()
print(f"   {'Grade':<8} {'Sequences':>10} "
      f"{'Train':>8} {'Val':>6} "
      f"{'Test':>6} {'Epochs':>8}")
print(f"   {'─'*55}")

for grade in GRADES:

    # ── 1. Get this grade's data ──────────────────────────
    gdf = (df[df['TeaGrade'] == grade]
           .sort_values('Date')
           .reset_index(drop=True))
    demand_raw = gdf['DemandKg'].values.reshape(-1, 1)

    # ── 2. Grade-specific scaler ──────────────────────────
    scaler = MinMaxScaler(feature_range=(0, 1))
    demand_scaled = scaler.fit_transform(
        demand_raw).flatten()

    # ── 3. Build sequences ────────────────────────────────
    X, y = build_sequences(demand_scaled,
                           LOOK_BACK, MAX_HORIZON)
    X = X.reshape(X.shape[0], X.shape[1], 1)

    # ── 4. Chronological split ────────────────────────────
    n = len(X)
    train_end = int(n * 0.75)
    val_end   = int(n * 0.88)

    X_train = X[:train_end]
    y_train = y[:train_end]
    X_val   = X[train_end:val_end]
    y_val   = y[train_end:val_end]
    X_test  = X[val_end:]
    y_test  = y[val_end:]

    print(f"   {grade:<8} {n:>10} "
          f"{len(X_train):>8} {len(X_val):>6} "
          f"{len(X_test):>6}", end='')

    # ── 5. Build model ────────────────────────────────────
    model = build_model(LOOK_BACK, MAX_HORIZON)

    # ── 6. Callbacks ──────────────────────────────────────
    es = EarlyStopping(
        monitor='val_loss',
        patience=PATIENCE,
        restore_best_weights=True,
        verbose=0
    )
    rlr = ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,
        min_lr=1e-6,
        verbose=0
    )

    # ── 7. Train ──────────────────────────────────────────
    hist = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[es, rlr],
        verbose=0          # silent — print only summary
    )
    epochs_run = len(hist.history['loss'])
    print(f" {epochs_run:>8}")

    # ── 8. Evaluate at 30/45/60 ───────────────────────────
    y_pred_scaled = model.predict(X_test, verbose=0)

    grade_results = {}
    for h in [30, 45, 60]:
        yt = scaler.inverse_transform(
            y_test[:, :h].reshape(-1, 1)
        ).reshape(-1, h)
        yp = scaler.inverse_transform(
            y_pred_scaled[:, :h].reshape(-1, 1)
        ).reshape(-1, h)

        grade_results[h] = {
            'MAE':  round(float(mean_absolute_error(
                yt.flatten(), yp.flatten())), 3),
            'RMSE': round(float(np.sqrt(mean_squared_error(
                yt.flatten(), yp.flatten()))), 3),
            'MAPE': round(float(mape_score(yt, yp)), 3),
        }

    all_grade_results[grade] = grade_results

    # ── 9. Save model and scaler ──────────────────────────
    model_path  = (f'saved_models/lstm_v2/'
                   f'lstm_{grade.lower()}.keras')
    scaler_path = (f'saved_models/lstm_v2/'
                   f'scaler_{grade.lower()}.pkl')

    model.save(model_path)
    joblib.dump(scaler, scaler_path)

    all_grade_metadata[grade] = {
        'model_path':  model_path,
        'scaler_path': scaler_path,
        'epochs':      epochs_run,
        'look_back':   LOOK_BACK,
        'max_horizon': MAX_HORIZON,
        'mape_30':     grade_results[30]['MAPE'],
        'mape_45':     grade_results[45]['MAPE'],
        'mape_60':     grade_results[60]['MAPE'],
    }

print()
print("="*65)
print("✅ ALL 5 MODELS TRAINED AND SAVED!")
print("="*65)

   TRAINING — ONE MODEL PER GRADE

   Grade     Sequences    Train    Val   Test   Epochs
   ───────────────────────────────────────────────────────
   BOP             976      732    126    118WARNING:tensorflow:From C:\Users\mrmra\Desktop\Smart tea\SmartTea_AI\smarttea_env\Lib\site-packages\keras\src\backend\common\global_state.py:82: The name tf.reset_default_graph is deprecated. Please use tf.compat.v1.reset_default_graph instead.

      105
   BOPF            976      732    126    118       64
   DUST            976      732    126    118       86
   FNGS            976      732    126    118       66
   OP              976      732    126    118       61

✅ ALL 5 MODELS TRAINED AND SAVED!


In [7]:
# ============================================================
# Full results: all grades, all horizons
# ============================================================

print()
print("="*70)
print("   FINAL RESULTS — ALL GRADES × ALL HORIZONS")
print("="*70)
print()

for horizon in [30, 45, 60]:
    print(f"   ── {horizon}-DAY HORIZON ──────────────────────────")
    print(f"   {'Grade':<8} {'MAE (kg)':>10} "
          f"{'RMSE (kg)':>12} {'MAPE (%)':>10}  Grade")
    print(f"   {'─'*55}")

    mapes = []
    for grade in GRADES:
        r    = all_grade_results[grade][horizon]
        mape = r['MAPE']
        mapes.append(mape)

        if mape < 10:
            g = '🌟 Excellent'
        elif mape < 15:
            g = '✅ Good'
        elif mape < 20:
            g = '⚠️  Acceptable'
        else:
            g = '❌ Poor'

        print(f"   {grade:<8} "
              f"{r['MAE']:>10.3f} "
              f"{r['RMSE']:>12.3f} "
              f"{mape:>10.3f}%  {g}")

    avg_mape = np.mean(mapes)
    print(f"   {'─'*55}")
    print(f"   {'AVERAGE':<8} {'':>10} "
          f"{'':>12} {avg_mape:>10.3f}%")
    print()

print()
print("   REFERENCE (old recursive LSTM):")
print("   BOP 30-day MAPE: ~7.3% (your original result)")
print()
print("   The new models should be close to this")
print("   across ALL grades and ALL horizons.")


   FINAL RESULTS — ALL GRADES × ALL HORIZONS

   ── 30-DAY HORIZON ──────────────────────────
   Grade      MAE (kg)    RMSE (kg)   MAPE (%)  Grade
   ───────────────────────────────────────────────────────
   BOP          20.634       25.315      6.694%  🌟 Excellent
   BOPF         17.300       22.030      6.368%  🌟 Excellent
   DUST         16.647       21.053      7.209%  🌟 Excellent
   FNGS         13.623       17.529      7.858%  🌟 Excellent
   OP           10.134       12.779      9.506%  🌟 Excellent
   ───────────────────────────────────────────────────────
   AVERAGE                               7.527%

   ── 45-DAY HORIZON ──────────────────────────
   Grade      MAE (kg)    RMSE (kg)   MAPE (%)  Grade
   ───────────────────────────────────────────────────────
   BOP          21.018       25.891      6.738%  🌟 Excellent
   BOPF         18.335       23.588      6.751%  🌟 Excellent
   DUST         17.939       22.607      7.749%  🌟 Excellent
   FNGS         14.474       18.538

In [8]:
# ============================================================
# Save one metadata file that the API will read
# ============================================================

metadata = {
    'version':           '2.1',
    'strategy':          'direct_multioutput',
    'look_back':         LOOK_BACK,
    'max_horizon':       MAX_HORIZON,
    'supported_horizons':[30, 45, 60],
    'grades':            GRADES,
    'per_grade':         all_grade_metadata,
    'architecture':      ('LSTM(64) → BN → Drop → '
                          'LSTM(32) → Drop → '
                          'Dense(32) → Dense(60)'),
}

joblib.dump(metadata,
            'saved_models/lstm_v2/metadata.pkl')

with open('saved_models/lstm_v2/metadata.json',
          'w') as f:
    json.dump(metadata, f, indent=2)

print("="*55)
print("   SAVED FILES")
print("="*55)
print()
for grade in GRADES:
    g = grade.lower()
    print(f"   ✅ lstm_{g}.keras")
    print(f"   ✅ scaler_{g}.pkl")

print()
print("   ✅ metadata.pkl")
print("   ✅ metadata.json")
print()
print("   Old model UNTOUCHED:")
print("   saved_models/lstm/ ← still there")
print()

# Print file sizes
for grade in GRADES:
    g    = grade.lower()
    path = f'saved_models/lstm_v2/lstm_{g}.keras'
    size = os.path.getsize(path) / 1024
    print(f"   {grade}: {size:.0f} KB")

print()
print("🎓 TRAINING COMPLETE!")
print()
print("   NEXT STEP:")
print("   Paste your Cell 6 results here")
print("   and confirm MAPE is < 15% before")
print("   updating the API")

   SAVED FILES

   ✅ lstm_bop.keras
   ✅ scaler_bop.pkl
   ✅ lstm_bopf.keras
   ✅ scaler_bopf.pkl
   ✅ lstm_dust.keras
   ✅ scaler_dust.pkl
   ✅ lstm_fngs.keras
   ✅ scaler_fngs.pkl
   ✅ lstm_op.keras
   ✅ scaler_op.pkl

   ✅ metadata.pkl
   ✅ metadata.json

   Old model UNTOUCHED:
   saved_models/lstm/ ← still there

   BOP: 430 KB
   BOPF: 430 KB
   DUST: 430 KB
   FNGS: 430 KB
   OP: 430 KB

🎓 TRAINING COMPLETE!

   NEXT STEP:
   Paste your Cell 6 results here
   and confirm MAPE is < 15% before
   updating the API
